In [2]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import yfinance as yf
import ta
import math
from fredapi import Fred
import logging
import ecbdata
import requests
import json

Fred_api = "e886df7269c2c4e6209754d4ea0371d5"
Grok_api = "xai-zBPdvxZOTUXY6Uaos1erYo0RIQultvO4E1TRvZ4eGhkNpV66ZYjH8Uke9Flf5iUp7xqg5SttSJyp9ccr"

xai_base_url = "https://api.x.ai/v1"
xai_headers = {
    "Authorization": f"Bearer {Grok_api}",
    "Content-Type": "application/json"
}

def query_grok(prompt, data=None):
    content = prompt + (f"\nData: {json.dumps(data, default=str)}" if data else "")
    payload = {
        "model": "grok-beta",
        "messages": [
            {"role": "user", "content": content}
        ],
        "max_tokens": 200
    }
    try:
        response = requests.post(f"{xai_base_url}/chat/completions", headers=xai_headers, json=payload)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Grok API error: {e}")
        return None

In [3]:
test_response = query_grok("Say 'hello' if you’re working.")
print(test_response)

Hello! How can I assist you today?


# List Of Tradeable Pairs And Indicators

In [4]:
# Initialize MetaTrader 5 connection
mt5.initialize()

# Updated list of currency pairs
pairs = [
    "EURUSD",  # Euro / US Dollar
    "EURCHF",  # Euro / Swiss Franc
    "EURJPY",  # Euro / Japanese Yen
    "USDCHF",  # US Dollar / Swiss Franc
    "CHFJPY",  # Swiss Franc / Japanese Yen
    
    "USDJPY"   # US Dollar / Japanese Yen
]

currencies = [
   "DX-Y.NYB", # Dollar Currency Index
    "^XDE",    # Euro Currency Index
    "^XDS",    # Chf Currency Index
    "^XDN"     # Yen Currency Index
]

# Function to get the latest Ask and Bid prices for a given pair
def get_latest_prices(symbol):
    # Get the latest tick data for the symbol
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Failed to get latest tick data for {symbol}")
        return None, None
    return tick.ask, tick.bid

# Function to get historical data for a given pair
def get_historical_data(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    # Fetch historical data
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, n_bars)
    if rates is None or len(rates) == 0:
        print(f"Failed to get historical data for {symbol}")
        return None
    data = pd.DataFrame(rates)
    data['time'] = pd.to_datetime(data['time'], unit='s')
    return data

# Function to calculate EMA, RSI, and ATR for a given pair
def calculate_indicators(symbol):
    # Get historical data for the pair
    data = get_historical_data(symbol)
    if data is None:
        return None, None, None, None

    # Calculate EMA 64
    data['EMA_64'] = ta.trend.ema_indicator(data['close'], window=64)

    # Calculate RSI 16
    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)

    # Calculate ATR 16
    data['ATR_16'] = ta.volatility.average_true_range(data['high'], data['low'], data['close'], window=16)

    # Get the latest values of the indicators
    latest_price = data['close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    latest_atr = data['ATR_16'].iloc[-1]

    return latest_price, latest_ema, latest_rsi, latest_atr

# Macroeconomic Data

In [5]:
import pandas as pd
from fredapi import Fred
import logging
from ecbdata import ecbdata

# Initialize FRED with your API key
fred = Fred(api_key=Fred_api)  # Replace with your actual API key

# Define the countries and their respective indicators
countries = {
    'USA': {'gdp': 'GDPC1', 'unemp': 'UNRATE', 'interest': 'FEDFUNDS', 'inflation': 'CPIAUCSL'},
    'Europe': {'gdp': 'CLVMEURSCAB1GQEA19', 'unemp': 'LRUNTTTTQZA156S', 'interest': 'IR3TIB01EZQ156N', 'inflation': 'CPHPTT01EZQ659N'},
    'Switzerland': {'gdp': 'CLVMNACSAB1GQCH', 'unemp': 'LRUNTTTTCHQ156S', 'interest': 'IR3TIB01CHQ156N', 'inflation': 'CHECPIALLMINMEI'},
    'Japan': {'gdp': 'JPNRGDPEXP', 'unemp': 'LRUNTTTTJPQ156S', 'interest': 'IR3TIB01JPQ156N', 'inflation': 'JPNCPIALLMINMEI'}
}


# Initialize empty dictionaries for storing the economic data
growth_rates = {}
unemp_rates = {}
interest_rates = {}
inflation_rates = {}
fallbacks = {
    'Europe': {
        'inflation': 'ICP.M.U2.N.000000.4.ANR',  # Alternative Europe inflation series
        'unemp': 'LFSI.M.U2.N.UNEHRT.TOTAL0.15_74.T',  # Alternative Europe unemployment series
    }
}

def get_10_years_Edata(symbol, country, indicator):
    try:
        economic_data = fred.get_series(symbol)
        economic_data = economic_data[economic_data.index >= pd.Timestamp.now() - pd.DateOffset(years=10)]
        if economic_data.empty:
            raise ValueError(f"No data returned for {symbol}")
        logging.info(f"Successfully fetched {indicator} data for {country} using {symbol} ({len(economic_data)} points)")
        return economic_data
    except Exception as e:
        logging.error(f"Error fetching {indicator} data for {country}: {e}")
        return None
def calculate_growth_rate(gdp_series):
    """Calculate annualized quarterly GDP growth rate"""
    if gdp_series is None or len(gdp_series) < 2:
        return None
    gdp_now = gdp_series.iloc[-1]
    gdp_previous = gdp_series.iloc[-2]
    return ((gdp_now - gdp_previous) / gdp_previous) * 100 * 4

def get_latest_value(series):
    """Get the most recent value from a series"""
    if series is None or len(series) < 1:
        return None
    return series.iloc[-1]

def calculate_inflation_rate(cpi_series):
    """Calculate year-over-year inflation rate"""
    if cpi_series is None or len(cpi_series) < 13:  # Need 12 months + 1 for monthly data
        logging.warning(f"Insufficient data for inflation calculation: {len(cpi_series)} points")
        return None
    cpi_now = cpi_series.iloc[-1]
    cpi_year_ago = cpi_series.iloc[-13]  # Assumes monthly data
    return ((cpi_now - cpi_year_ago) / cpi_year_ago) * 100

# Update the economic data and store it in a DataFrame
def update_economic_data():
    global growth_rates, unemp_rates, interest_rates, inflation_rates
    for country, symbols in countries.items():
        # GDP Growth
        gdp_data = get_10_years_Edata(symbols['gdp'], country, 'gdp')
        if gdp_data is not None:
            growth_rates[country] = calculate_growth_rate(gdp_data)
        
        # Unemployment
        unemp_data = get_10_years_Edata(symbols['unemp'], country, 'unemployment')
        if unemp_data is not None:
            unemp_rates[country] = get_latest_value(unemp_data)
            
        
        # Interest Rates
        interest_data = get_10_years_Edata(symbols['interest'], country, 'interest')
        if interest_data is not None:
            interest_rates[country] = get_latest_value(interest_data)
        
        # Inflation
        inflation_data = get_10_years_Edata(symbols['inflation'], country, 'inflation')
        if inflation_data is not None:
            if country == 'Europe' and symbols['inflation'] == 'CPHPTT01EZQ659N':
                inflation_rates[country] = get_eur_inflation()
            else:
                inflation_rates[country] = calculate_inflation_rate(inflation_data)
          
def get_eur_inflation():
    economic_data = ecbdata.get_series('ICP.M.U2.N.000000.4.ANR', start='2024-01')
    latest = economic_data['OBS_VALUE'].iloc[-1]  # Fixed: Use 'economic_data'
    return latest

def get_eur_unemployment():
    economic_data = ecbdata.get_series('LFSI.M.U2.N.UNEHRT.TOTAL0.15_74.T', start='2024-01')
    latest = economic_data['OBS_VALUE'].iloc[-1]  # Fixed: Use 'economic_data'
    return latest

# Create a DataFrame to hold the economic data
def create_economic_dataframe():
    update_economic_data()
    
    # Combine the data into a DataFrame
    data = {
        'GDP Growth': growth_rates,
        'Unemployment': unemp_rates,
        'Interest Rate': interest_rates,
        'Inflation Rate': inflation_rates
    }
    
    df = pd.DataFrame(data)
    df.loc['Japan', "Inflation Rate"] = 4
    df.loc['Europe', 'Unemployment'] = get_eur_unemployment() 
    df.loc['Europe', 'Inflation Rate'] =get_eur_inflation()
    return df
df = create_economic_dataframe()

df


ERROR:root:Error fetching unemployment data for Europe: Bad Request.  The series does not exist.
ERROR:root:Error fetching inflation data for Europe: No data returned for CPHPTT01EZQ659N


,GDP Growth,Unemployment,Interest Rate,Inflation Rate
USA,2.232707,4.000000,4.330000,2.999413
Europe,0.203240,6.268223,2.996487,2.500000
Switzerland,1.536775,4.497941,0.767250,0.401528
Japan,1.233826,2.466667,0.334667,4.000000


# Currency Index Data

In [6]:
def get_currency_data(currency):
    # Fetch DXY historical data (last 10 days, 15m interval)
    dxy = yf.Ticker(currency)
    data = dxy.history(period="10d", interval="15m")
    
    # Recalculate RSI & EMA
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=64)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=196)

    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

def get_currencies_table(currencies):
    dict = {}
    for i in currencies:
        name = i
        if i == "DX-Y.NYB":
            name = "USD"
        elif i == "^XDE" :
            name = "EUR"
        elif i== "^XDS" :
            name = "CHF"
        elif i== "^XDN" :
            name = "JPY"
        
        latest_price, latest_ema, latest_rsi = get_currency_data(i)
        dict.update({name: [latest_price, latest_ema, latest_rsi ]})
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Price', 'EMA', 'RSI'])
    return data

get_currencies_table(currencies)

,Price,EMA,RSI
USD,106.696999,106.581471,54.573389
EUR,104.632004,104.346047,52.342564
CHF,111.395798,110.786601,57.048809
JPY,66.988899,66.223327,61.864333


# Pip Value

In [7]:
def get_pip_value(symbol):
    dec = 0.0001
    if "JPY" in symbol:
        dec = 0.01
    latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(symbol)
    pip_value = (dec * 100000) / latest_price
    return pip_value

# Position Size

In [8]:
def get_position_size(pair, stop_loss):
    account_info = mt5.account_info()
    balance = account_info.balance
    risk_amount = 0.005 * balance
    size = risk_amount / ( stop_loss * get_pip_value(pair))
    size = round(size, 2)
    return size

# Bias

In [9]:
# from imfdatapy.imf import IMF
# growth_rates, unemp_rates, interest_rates, inflation_rates

def update_debt_to_gdp():
    # """
    # Fetch debt-to-GDP ratios for 2025 (latest projection) and 2024 from IMF WEO for EU, US, Japan, and Switzerland.
    # Returns a dictionary with 'latest' (2025) and 'previous' (2024) sub-dictionaries for comparison.
    # """
    # try:
    #     imf = IMF()
    #     # Fetch "General government gross debt" (% of GDP) for EA (Euro Area), US, JP, CH
    #     debt_data = imf.get_series("WEO", "GGXWDG_NGDP", country=["EA", "US", "JP", "CH"], period="A")
        
    #     # Explicitly target 2025 (latest projection) and 2024 (previous year)
    #     debt_to_gdp = {
    #         "latest": {
    #             "EU": debt_data["EA"].loc[2025]["value"],
    #             "US": debt_data["US"].loc[2025]["value"],
    #             "Japan": debt_data["JP"].loc[2025]["value"],
    #             "Switzerland": debt_data["CH"].loc[2025]["value"]
    #         },
    #         "previous": {
    #             "EU": debt_data["EA"].loc[2024]["value"],
    #             "US": debt_data["US"].loc[2024]["value"],
    #             "Japan": debt_data["JP"].loc[2024]["value"],
    #             "Switzerland": debt_data["CH"].loc[2024]["value"]
    #         }
    #     }
    #     print(f"Debt-to-GDP fetched: Latest (2025 projection) vs Previous (2024)")
    #     return debt_to_gdp
    # except Exception as e:
    #     print(f"Error fetching debt-to-GDP data: {e}")
    #     # Fallback values for 2025 (projections) and 2024 based on trends from prior data
    return {
        "latest": {"EU": 89.0, "US": 122.0, "Japan": 261.0, "Switzerland": 43.5},  # 2025 projections
        "previous": {"EU": 88.5, "US": 120.0, "Japan": 260.0, "Switzerland": 43.3}  # 2024 estimates
    }

# Initialize debt_to_gdp dynamically
debt_to_gdp = update_debt_to_gdp()

def compare_economies(country1, country2):
    """
    Compare the economic data between two countries using latest data and year-over-year debt trends.
    Returns a 'buy' signal for country1 or 'sell' for country2 based on macroeconomic performance.
    Uses 5 factors, with debt-to-GDP trend as a tiebreaker, and a 2.5 threshold to minimize neutral outcomes.
    """
    buy_factors = 0
    sell_factors = 0
    
    # Compare GDP Growth Rates (weight: 1.5)
    if growth_rates.get(country1, 0) > growth_rates.get(country2, 0):
        buy_factors += 1.5
    elif growth_rates.get(country1, 0) < growth_rates.get(country2, 0):
        sell_factors += 1.5
    
    # Compare Unemployment Rates (lower is better, weight: 1)
    if unemp_rates.get(country1, float('inf')) < unemp_rates.get(country2, float('inf')):
        buy_factors += 1
    elif unemp_rates.get(country1, float('inf')) > unemp_rates.get(country2, float('inf')):
        sell_factors += 1
    
    # Compare Interest Rates (higher is generally better, weight: 1)
    if interest_rates.get(country1, 0) > interest_rates.get(country2, 0):
        buy_factors += 1
    elif interest_rates.get(country1, 0) < interest_rates.get(country2, 0):
        sell_factors += 1
    
    # Compare Inflation Rates (lower is better, weight: 1.5)
    if inflation_rates.get(country1, float('inf')) < inflation_rates.get(country2, float('inf')):
        buy_factors += 1.5
    elif inflation_rates.get(country1, float('inf')) > inflation_rates.get(country2, float('inf')):
        sell_factors += 1.5
    
    # Compare Debt-to-GDP Trend (lower latest vs previous is better, weight: 0.5)
    debt_trend1 = debt_to_gdp["latest"].get(country1, float('inf')) - debt_to_gdp["previous"].get(country1, float('inf'))
    debt_trend2 = debt_to_gdp["latest"].get(country2, float('inf')) - debt_to_gdp["previous"].get(country2, float('inf'))
    if debt_trend1 < debt_trend2:  # Smaller increase or larger decrease favors country1
        buy_factors += 0.5
    elif debt_trend1 > debt_trend2:
        sell_factors += 0.5
    
    # Determine the final bias with a 2.5 threshold to minimize neutral outcomes
    if buy_factors > 2.5:
        return 'buy'
    elif sell_factors > 2.5:
        return 'sell'
    else:
        return 'neutral'

def bias_for_pairs(pairs):
    bias_results = {}
    
    # Update debt_to_gdp before running comparisons
    global debt_to_gdp
    debt_to_gdp = update_debt_to_gdp()
    
    # Loop through each pair and get the macroeconomic comparison
    for i in pairs:
        if "EUR" in i and "USD" in i:
            bias = compare_economies("EU", "US")  # Compare Eurozone with USA
        elif "EUR" in i and "CHF" in i:
            bias = compare_economies("EU", "Switzerland")  # Compare Eurozone with Switzerland
        elif "EUR" in i and "JPY" in i:
            bias = compare_economies("EU", "Japan")  # Compare Eurozone with Japan
        elif "USD" in i and "CHF" in i:
            bias = compare_economies("US", "Switzerland")  # Compare USA with Switzerland
        elif "CHF" in i and "JPY" in i:
            bias = compare_economies("Switzerland", "Japan")  # Compare Switzerland with Japan
        elif "USD" in i and "JPY" in i:
            bias = compare_economies("US", "Japan")  # Compare USA with Japan
        
        # Update the dictionary with the bias result
        bias_results[i] = bias
    
    return bias_results


# Sentiment 

In [10]:
def get_sentiment(pair):
    table = get_currencies_table(currencies)
    signal1 = "neutral"
    signal2 = "neutral"
    sentiment = "neutral"
    for index in table.index:
        if pair.startswith(index):
            currency1 = index
        elif pair.endswith(index):
            currency2 = index
    if table.loc[currency1, "Price"] > table.loc[currency1, "EMA"] and table.loc[currency1, "RSI"] > 50:
        signal1 = "buy"
    elif table.loc[currency1, "Price"] < table.loc[currency1, "EMA"] and table.loc[currency1, "RSI"] < 50:
        signal1 = "sell"
    if table.loc[currency2, "Price"] > table.loc[currency2, "EMA"] and table.loc[currency2, "RSI"] > 50:
        signal2 = "buy"
    elif table.loc[currency2, "Price"] < table.loc[currency2, "EMA"] and table.loc[currency2, "RSI"] < 50:
        signal2 = "sell"
    if signal1 == "buy" and signal2 == "sell":
        sentiment = "buy"
    elif signal1 == "sell" and signal2 == "buy":
        sentiment = "sell"
    return sentiment

for i in pairs:
    print(i, get_sentiment(i))
    print()

EURUSD neutral

EURCHF neutral

EURJPY neutral

USDCHF neutral

CHFJPY neutral

USDJPY neutral



# Signals

In [11]:
def detect_rsi_divergence(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    # Fetch historical data
    data = get_historical_data(symbol, timeframe, n_bars)
    if data is None or len(data) < 20:
        return "neutral"

    # Calculate RSI (16-period)
    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)

    # Identify peaks and troughs in price and RSI (5-bar window)
    data['price_high'] = data['close'][(data['close'] > data['close'].shift(1)) & 
                                       (data['close'] > data['close'].shift(-1)) & 
                                       (data['close'] > data['close'].shift(2)) & 
                                       (data['close'] > data['close'].shift(-2))]
    data['price_low'] = data['close'][(data['close'] < data['close'].shift(1)) & 
                                      (data['close'] < data['close'].shift(-1)) & 
                                      (data['close'] < data['close'].shift(2)) & 
                                      (data['close'] < data['close'].shift(-2))]
    data['rsi_high'] = data['RSI_16'][(data['RSI_16'] > data['RSI_16'].shift(1)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(-1)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(2)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(-2))]
    data['rsi_low'] = data['RSI_16'][(data['RSI_16'] < data['RSI_16'].shift(1)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(-1)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(2)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(-2))]

    # Get the most recent two highs and lows
    price_highs = data['price_high'].dropna().tail(2)
    price_lows = data['price_low'].dropna().tail(2)
    rsi_highs = data['rsi_high'].dropna().tail(2)
    rsi_lows = data['rsi_low'].dropna().tail(2)

    if len(price_highs) < 2 or len(price_lows) < 2 or len(rsi_highs) < 2 or len(rsi_lows) < 2:
        return "neutral"

    ph1, ph2 = price_highs.iloc[0], price_highs.iloc[1]
    pl1, pl2 = price_lows.iloc[0], price_lows.iloc[1]
    rh1, rh2 = rsi_highs.iloc[0], rsi_highs.iloc[1]
    rl1, rl2 = rsi_lows.iloc[0], rsi_lows.iloc[1]

    # Regular Divergence (Reversal)
    if pl2 < pl1 and rl2 > rl1 and rl2 < 40:
        return "buy"
    elif ph2 > ph1 and rh2 < rh1 and rh2 > 60:
        return "sell"

    # Hidden Divergence (Continuation)
    if pl2 > pl1 and rl2 < rl1 and rl2 < 50:
        return "buy"
    elif ph2 < ph1 and rh2 > rh1 and rh2 > 50:
        return "sell"

    return "neutral"

def detect_ict_signal(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    data = get_historical_data(symbol, timeframe, n_bars)
    if data is None or len(data) < 20:
        return "neutral"

    def is_buy_order_block(i):
        if data['close'][i] > data['open'][i] * 1.02:
            for j in range(1, 3):
                if data['low'][i+j] < data['low'][i]:
                    return False
            return True
        return False

    def is_sell_order_block(i):
        if data['close'][i] < data['open'][i] * 0.98:
            for j in range(1, 3):
                if data['high'][i+j] > data['high'][i]:
                    return False
            return True
        return False

    buy_blocks = [data['close'][i] for i in range(len(data)-3) if is_buy_order_block(i)]
    sell_blocks = [data['close'][i] for i in range(len(data)-3) if is_sell_order_block(i)]

    current_price = data['close'][0]
    pip_value = get_pip_value(symbol)
    threshold_pips = 10

    for block in buy_blocks:
        if abs(current_price - block) < threshold_pips * pip_value:
            return "buy"

    for block in sell_blocks:
        if abs(current_price - block) < threshold_pips * pip_value:
            return "sell"

    return "neutral"

def get_combined_signal(symbol):
    rsi_signal = detect_rsi_divergence(symbol)
    ict_signal = detect_ict_signal(symbol)
    
    if rsi_signal == ict_signal and rsi_signal in ["buy", "sell"]:
        return rsi_signal
    return "neutral"

# Forex Pairs Data table

In [12]:
def get_data_table(pairs):
    dict = {}
    bias_results = bias_for_pairs(pairs)
    for i in pairs:
        ask, bid = get_latest_prices(i)
        latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(i)
        dict.update({i: [ask,bid,latest_price, latest_ema, latest_rsi, latest_atr]})
    # Convert to DataFrame
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Ask', 'Bid', 'price', 'EMA', 'RSI', 'ATR'])
    
    data['Stop Loss']= data['ATR'] * 4
    data['Take Profit'] = data['Stop Loss'] * 2
    for i in pairs:
        dec = 10000
        if "JPY" in i:
            dec = 100
        stop_loss = round(data.loc[i, "Stop Loss"] * dec)
        take_profit = round(data.loc[i, "Take Profit"] * dec)
        data.loc[i,'Round Stop Loss'] = int(stop_loss)
        data.loc[i,'Round Take Profit'] = int(take_profit)
        data.loc[i,'Pip Value'] = get_pip_value(i)
        data.loc[i,'Position Size'] = get_position_size(i, stop_loss)
        data.loc[i, 'Bias'] = bias_results.get(i)
        data.loc[i,'Sentiment'] = get_sentiment(i)
        data.loc[i,'Signal'] = get_combined_signal(i)
    return data
        
data = get_data_table(pairs)

data


,Ask,Bid,price,EMA,RSI,ATR,Stop Loss,Take Profit,Round Stop Loss,Round Take Profit,Pip Value,Position Size,Bias,Sentiment,Signal
EURUSD,1.04639,1.04637,1.04637,1.048153,39.463701,0.000685,0.002741,0.005481,27.0,55.0,9.556849,3.10,neutral,neutral,neutral
EURCHF,0.93931,0.93925,0.93925,0.941244,28.583198,0.000483,0.001933,0.003866,19.0,39.0,10.647019,3.95,sell,neutral,neutral
EURJPY,156.19100,156.17800,156.17800,156.688922,29.566350,0.161911,0.647643,1.295286,65.0,130.0,6.403032,1.92,sell,neutral,neutral
USDCHF,0.89767,0.89763,0.89763,0.898012,39.017374,0.000552,0.002208,0.004415,22.0,44.0,11.140075,3.26,sell,neutral,neutral
CHFJPY,166.28800,166.27100,166.27100,166.462227,38.038504,0.143791,0.575166,1.150331,58.0,115.0,6.014025,2.29,buy,neutral,neutral
USDJPY,149.26900,149.26300,149.26300,149.493084,33.991740,0.141523,0.566091,1.132183,57.0,113.0,6.699315,2.09,sell,neutral,neutral


# Decision Taking Order Code

In [ ]:
import time
from datetime import datetime

# Configurable FTMO defaults (adjustable for other firms)
MAX_DAILY_LOSS_PCT = 0.05  # 5%
MAX_TOTAL_LOSS_PCT = 0.10  # 10%

if not mt5.initialize():
    print("MT5 initialization failed. Exiting.")
    exit()

signal_history = {pair: "neutral" for pair in pairs}
INITIAL_ACCOUNT_SIZE = None  # Set once at start
LEVERAGE = None  # Fetched dynamically
realized_profit = 0  # Track closed trade profits

def initialize_account_params():
    """Fetch and lock initial account parameters from MT5."""
    global INITIAL_ACCOUNT_SIZE, LEVERAGE
    account_info = mt5.account_info()
    if not account_info:
        print("Failed to get account info. Using defaults.")
        INITIAL_ACCOUNT_SIZE = 160000
        LEVERAGE = 100
    else:
        INITIAL_ACCOUNT_SIZE = account_info.balance
        LEVERAGE = account_info.leverage
    print(f"Initialized: Initial Account Size = ${INITIAL_ACCOUNT_SIZE}, Leverage = {LEVERAGE}:1")

# Set initial params on first run
if INITIAL_ACCOUNT_SIZE is None:
    initialize_account_params()

def get_account_state():
    """Fetch current account state from MT5."""
    account_info = mt5.account_info()
    if not account_info:
        print("Failed to get account info.")
        return None, None
    return account_info.balance, account_info.equity

def get_ftmo_position_size(pair, stop_loss_pips):
    """Calculate position size with FTMO fixed limits and profit-adjusted risk."""
    balance, equity = get_account_state()
    if balance is None or equity is None:
        return 0.0
    
    max_daily_loss = INITIAL_ACCOUNT_SIZE * MAX_DAILY_LOSS_PCT  # Fixed to initial
    max_total_loss = INITIAL_ACCOUNT_SIZE * MAX_TOTAL_LOSS_PCT  # Fixed to initial
    
    daily_loss = max(0, balance - equity)  # Floating losses
    total_loss = INITIAL_ACCOUNT_SIZE - equity  # Total drawdown from start
    
    if daily_loss >= max_daily_loss:
        print(f"{pair} - Daily loss limit ({max_daily_loss:.2f}) reached.")
        return 0.0
    if total_loss >= max_total_loss:
        print(f"{pair} - Total loss limit ({max_total_loss:.2f}) reached.")
        return 0.0
    
    pip_decimal = 0.0001 if "JPY" not in pair else 0.01
    latest_price, _, _, _ = calculate_indicators(pair)
    if latest_price is None:
        return 0.0
    pip_value = (pip_decimal * 100000) / latest_price
    
    # Adjust risk based on profits: 0.5% base, reduced if profits > 10%
    base_risk_pct = 0.005
    if realized_profit > 0.10 * INITIAL_ACCOUNT_SIZE:  # After 10% profit
        base_risk_pct = 0.0025  # Halve risk to secure gains
    risk_per_trade = balance * base_risk_pct
    available_daily_risk = max_daily_loss - daily_loss
    available_total_risk = max_total_loss - total_loss
    risk_amount = min(risk_per_trade, available_daily_risk, available_total_risk)
    
    size = risk_amount / (stop_loss_pips * pip_value)
    size = round(size, 2)
    
    potential_loss = size * stop_loss_pips * pip_value
    if potential_loss > available_daily_risk or (total_loss + potential_loss) > max_total_loss:
        size = min(available_daily_risk, available_total_risk) / (stop_loss_pips * pip_value)
        size = round(size, 2)
        print(f"{pair} - Adjusted to {size} lots to fit limits.")
    
    return size

# ... (previous imports and functions unchanged up to calculate_score) ...

def calculate_score(bias, sentiment, combined_signal, pair, data_row):
    bias_score = map_to_score(bias, 'bias')
    sentiment_score = map_to_score(sentiment, 'sentiment')
    signal_score = map_to_score(combined_signal, 'signal')

    pair_data = {
        "pair": pair,
        "price": float(data_row["price"]),
        "EMA": float(data_row["EMA"]),
        "RSI": float(data_row["RSI"]),
        "ATR": float(data_row["ATR"]),
        "bias": bias,
        "sentiment": sentiment,
        "signal": combined_signal
    }
    
    prompt = "Analyze this forex pair data and return only 'buy', 'sell', or 'neutral' based on your assessment."
    grok_response = query_grok(prompt, pair_data)
    if grok_response not in ['buy', 'sell', 'neutral']:
        print(f"Grok failed for {pair}: {grok_response}, defaulting to RSI-based logic")
        # Fallback: If RSI < 40, "buy"; > 60, "sell"; else "neutral"
        rsi = float(data_row["RSI"])
        grok_response = "buy" if rsi < 40 else "sell" if rsi > 60 else "neutral"
    grok_score = map_to_score(grok_response, 'grok')
    
    news = get_news_score(pair)
    news_score = map_to_score(news, 'news')
    
    final_score = (0.25 * bias_score) + (0.20 * sentiment_score) + (0.15 * signal_score) + (0.20 * grok_score) + (0.20 * news_score)
    
    print(f"{pair} - Bias: {bias_score} (0.25), Sentiment: {sentiment_score} (0.20), Signal: {signal_score} (0.15), Grok: {grok_score} (0.20), News: {news_score} (0.20), Final score: {final_score:.2f}, Grok Response: {grok_response}")
    
    return final_score

# ... (rest of Cell 14 unchanged from last version) ...

def adjust_position_size(base_size, score):
    if abs(score) >= 0.70:
        adjusted_size = base_size * 2
        print(f"Strong alignment detected, doubling position size to {adjusted_size:.2f}")
        return adjusted_size
    return base_size

def is_market_open(symbol):
    tick = mt5.symbol_info_tick(symbol)
    if tick and tick.time:
        last_tick_time = datetime.fromtimestamp(tick.time)
        current_time = datetime.now()
        return (current_time - last_tick_time).total_seconds() < 300
    return False

def send_order(symbol, action, volume, stop_loss, take_profit):
    if not mt5.initialize():
        return False, "MT5 initialization failed"
    if not is_market_open(symbol):
        return False, f"Cannot trade {symbol}: Market is closed"
    symbol_info = mt5.symbol_info(symbol)
    if not symbol_info:
        return False, f"Failed to get symbol info for {symbol}"
    
    tick_size = symbol_info.point
    min_stop_distance = symbol_info.trade_stops_level * tick_size
    
    price = mt5.symbol_info_tick(symbol).ask if action in ['buy', 'strong buy'] else mt5.symbol_info_tick(symbol).bid
    sl_price = price - stop_loss if action in ['buy', 'strong buy'] else price + stop_loss
    tp_price = price + take_profit if action in ['buy', 'strong buy'] else price - take_profit
    
    if abs(price - sl_price) < min_stop_distance:
        stop_loss = min_stop_distance * 1.5
        sl_price = price - stop_loss if action in ['buy', 'strong buy'] else price + stop_loss
    if abs(price - tp_price) < min_stop_distance:
        take_profit = min_stop_distance * 3
        tp_price = price + take_profit if action in ['buy', 'strong buy'] else price - take_profit
    
    order_type = mt5.ORDER_TYPE_BUY if action in ['buy', 'strong buy'] else mt5.ORDER_TYPE_SELL
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": round(volume, 2),
        "type": order_type,
        "price": price,
        "sl": sl_price,
        "tp": tp_price,
        "deviation": 10,
        "magic": 234000,
        "comment": "Automated Trade",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        return False, f"Order failed for {symbol}: {result.comment}"
    global realized_profit
    # Simplified profit estimate (assuming TP hit)
    pip_value = get_pip_value(symbol)
    profit_pips = (data.loc[symbol, 'Round Take Profit'] - stop_loss_pips) if action in ['buy', 'strong buy'] else (stop_loss_pips - data.loc[symbol, 'Round Take Profit'])
    trade_profit = profit_pips * pip_value * volume
    realized_profit += trade_profit
    return True, f"{symbol} {action} {volume} lots, Score: {score:.2f}, Est. Profit: ${trade_profit:.2f}"

last_summary_time = None
actions_taken = []

while True:
    try:
        current_time = datetime.now()
        data = get_data_table(pairs)
        market_open = False
    
        for pair in pairs:
            bias = data.loc[pair, 'Bias']
            sentiment = data.loc[pair, 'Sentiment']
            combined_signal = get_combined_signal(pair)
            stop_loss_pips = data.loc[pair, 'Round Stop Loss']
            stop_loss = stop_loss_pips / (10000 if "JPY" not in pair else 100)
            take_profit = data.loc[pair, 'Round Take Profit'] / (10000 if "JPY" not in pair else 100)
            
            if is_market_open(pair):
                market_open = True
                last_signal = signal_history[pair]
                if combined_signal != last_signal and combined_signal != "neutral":
                    score = calculate_score(bias, sentiment, combined_signal, pair, data.loc[pair])
                    base_size = get_ftmo_position_size(pair, stop_loss_pips)
                    if base_size <= 0:
                        continue
                    
                    if score <= -0.70:
                        action = 'strong sell'
                    elif score < 0:
                        action = 'sell'
                    elif score == 0:
                        action = None
                    elif score <= 0.70:
                        action = 'buy'
                    else:
                        action = 'strong buy'
                    
                    if action:
                        adjusted_size = adjust_position_size(base_size, score)
                        if adjusted_size > 0:
                            success, message = send_order(pair, action, adjusted_size, stop_loss, take_profit)
                            if success:
                                actions_taken.append(f"Trade executed: {message}")
                                signal_history[pair] = combined_signal
                            else:
                                actions_taken.append(f"Failed: {message}")

        if last_summary_time is None or (current_time - last_summary_time).total_seconds() >= 1800:
            balance, equity = get_account_state()
            if balance is None:
                balance = INITIAL_ACCOUNT_SIZE
            if not market_open:
                print(f"30-minute summary at {current_time}: Market closed - no trading possible")
            else:
                print(f"30-minute summary at {current_time}:")
                if actions_taken:
                    for action in actions_taken:
                        print(f"  {action}")
                else:
                    print("  No actions taken")
                print(f"Balance: ${balance:.2f}, Equity: ${equity:.2f}, Realized Profit: ${realized_profit:.2f}")
            actions_taken = []
            last_summary_time = current_time
    
    except Exception as e:
        print(f"Error in cycle: {e}")
    
    time.sleep(60)

Initialized: Initial Account Size = $159977.49, Leverage = 100:1
30-minute summary at 2025-02-24 17:05:23.453736:
  No actions taken
Balance: $159977.49, Equity: $160528.33, Realized Profit: $0.00
